# Reproducing Gu et al. (2026) from dandiset 001836

Paper. Gu S, Liu C, Gillespie AK, Nevers R, Denovellis EL, Coulter ME, Kay K, Frank LM (2026).
Naturalistic behavior and self-generated neural activity predictive of self-correction. bioRxiv,
[doi:10.64898/2026.05.26.727951](https://doi.org/10.64898/2026.05.26.727951). The claims file
(`claims/001836/10.64898-2026.05.26.727951.yaml`) was written from the bioRxiv preprint, version 1
(May 27, 2026), with its supplement (Materials and Methods, Figures S1 to S12).

Dandiset. [DANDI:001836](https://dandiarchive.org/dandiset/001836/0.261030.0428), version
0.261030.0428. The paper cites the dandiset without a version. Two versions are published,
0.260526.1804 and 0.261030.0428; the cell below shows that they hold the same 2494 assets with the
same blobs, so they differ only in metadata. The most recent one is pinned, as the protocol asks
when the paper cites none.

What is tested. The dandiset holds the raw recordings (30 kHz voltage, inertial sensors, LED
tracking, well pokes and pumps, Statescript logs), a 1 kHz LFP file per day, and Spyglass analysis
files: head position, linearized position, and decodes triggered on events. It does not hold the
decoder's posterior, the spike marks, the detected remote or local extended events, theta cycles or
ripples. The behavioral claims (Figure 1, Figure S1F) are tested; the claims on hippocampal content
need the decoder output, which would have to be recomputed from 12.7 TB of raw voltage, and are
blocked.

| Claim | Status |
|---|---|
| fig1c-learning | tested |
| fig1e-trial-counts | tested |
| fig1d-initial-switch-bias | tested |
| fig1d-switch-bias-declines | tested |
| figs1f-multiple-com | tested |
| fig1g-com-corrective | tested |
| fig1g-com-matches-run-through | tested |
| fig1h-final-choice-less-switching | tested |
| fig1h-final-ratio-near-one | tested |
| fig3b-extended-before-stopping, fig3b-distance-predicts-com, fig3c-distant-after-stopping, fig3d-distant-vs-inbound | blocked (metadata_insufficient) |
| figs6, figs7b, figs3, figs11b, fig4a (two), fig4b, figs11a, fig4c, fig4d, fig4e, figs11c, fig5a, fig5b, figs8, figs10 | blocked (too_large) |

In [ ]:
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy import stats

import comtask as ct

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 300)
warnings.filterwarnings("ignore", category=RuntimeWarning)

started = time.time()
assets = ct.list_assets()
print(f"dandiset {ct.DANDISET_ID} version {ct.DANDISET_VERSION}: {len(assets)} assets, "
      f"{assets['size'].sum() / 1e12:.2f} TB")
R = ct.load_all(assets)
print(f"loaded {len(R)} rat-days in {time.time() - started:.0f} s")

## Data check

In [ ]:
# The two published versions: same paths and blobs?
other = ct._get(f"{ct.API}/dandisets/{ct.DANDISET_ID}/versions/")
versions = [v["version"] for v in other["results"] if v["version"] != "draft"]
print("published versions:", versions)
for v in versions:
    if v == ct.DANDISET_VERSION:
        continue
    url = f"{ct.API}/dandisets/{ct.DANDISET_ID}/versions/{v}/assets/?page_size=1000"
    blobs = {}
    while url:
        page = ct._get(url)
        blobs.update({a["path"]: a["blob"] for a in page["results"]})
        url = page["next"]
    same = sum(blobs.get(p) == b for p, b in zip(assets.path, assets.blob))
    print(f"version {v}: {len(blobs)} assets, {same} with the same path and blob as {ct.DANDISET_VERSION}")

print("\nAssets per rat and kind:")
print(pd.crosstab(assets.rat, assets.kind).to_string())
print("\nDays per rat:", assets.groupby("rat").day.nunique().to_dict())
kinds = pd.DataFrame({d: r["obj_counts"] for d, r in R.items()}).T.fillna(0).astype(int)
kinds["rat"] = [d.split("-")[0] for d in kinds.index]
print("\nAnalysis files (obj-*.nwb, not *_behavior.nwb) by content, per rat:")
print(kinds.groupby("rat").sum().to_string())
print("\n500 Hz and 23 Hz linearizations per rat:",
      {rat: (sum(R[d]["n_lin_500"] for d in R if d.startswith(rat)),
             sum(R[d]["n_lin_23"] for d in R if d.startswith(rat))) for rat in ct.RATS})

Each run epoch has two linearizations, at the camera rate (about 23 Hz) and upsampled to 500 Hz.
The 500 Hz one is used (Methods, Change of mind time detection). Rat M's days hold each 500 Hz
linearization twice; the cell below compares the two copies.

In [ ]:
dups = pd.DataFrame([x for r in R.values() for x in r["dups"]])
print(f"run epochs with more than one 500 Hz linearization: {len(dups)}")
if len(dups):
    dups["rat"] = dups.file_a.str.slice(4, 9)
    print(dups.groupby("rat").identical.agg(["size", "sum"]).rename(columns={"size": "pairs", "sum": "identical"}).to_string())
print("\nStatescript log names (rat M's are spelled 'syayescript'):")
print({d.split("-")[0]: sorted({n.split(" ")[0] for n in R[d]["statescript_names"]}) for d in R})

In [ ]:
checks = pd.concat([r["checks"] for r in R.values()], ignore_index=True)
print("Run epochs per rat and day, and trials built from the digital events:")
print(checks.groupby(["rat", "day"]).agg(sessions=("epoch", "size"), trials=("trials", "sum"),
                                          rewarded=("rewarded", "sum"),
                                          ss_outer_rewards=("ss_outer_rewards", "sum"),
                                          ss_outerCount=("ss_outerCount", "sum")).to_string())
ss = checks[checks.ss_outerCount > 0]
print(f"\nAgainst the Statescript logs (rats E, J, K, L; {len(ss)} sessions): rewarded trials equal the "
      f"logged outer rewards in {np.mean(ss.rewarded == ss.ss_outer_rewards):.3f} of sessions "
      f"(largest difference {np.abs(ss.rewarded - ss.ss_outer_rewards).max()}); trials equal the logged home "
      f"rewards in {np.mean(ss.trials == ss.ss_home_rewards):.3f} (largest difference "
      f"{np.abs(ss.trials - ss.ss_home_rewards).max()}); the logged outerCount exceeds the trials by a median of "
      f"{np.median(ss.ss_outerCount - ss.trials):.0f}.")
print("Sessions with fewer than 20 trials:")
print(checks[checks.trials < ct.MIN_TRIALS][["rat", "day", "epoch", "trials"]].to_string(index=False))

Clock check. Each outer well's track segment is read from where the head is at that well's
pokes; at a trial's final poke the head should be on the final arm's segment, and 10 s before
the poke (on the way out from home) it should not. The contrast is the check that the digital
events and the linearized position share a clock.

In [ ]:
print("Segment of each outer well (from pokes), per rat-day:")
print(pd.DataFrame({d: r["seg_of_arm"] for d, r in R.items()}).T.drop_duplicates().to_string())
seg_frac = pd.DataFrame([{"day": d, "arm": a, "frac": v["frac"]} for d, r in R.items()
                         for e in r["segmap"].values() for a, v in e.items()])
print(f"fraction of a well's pokes made on its modal segment, per epoch: median {seg_frac.frac.median():.3f}, "
      f"min {seg_frac.frac.min():.3f}")
print("\nFraction of trials with the head on the final arm's segment, at the final poke and shifted by 10 s:")
print(checks.groupby("rat")[["end_on_final_arm", "on_final_arm_10s_before", "on_final_arm_10s_after",
                             "median_depth_at_poke"]].mean().round(3).to_string())
lens = pd.DataFrame({d: r["arm_len"] for d, r in R.items()}).T
print(f"\nOuter arm length in linearized units (cm): min {lens.min().min():.1f}, max {lens.max().max():.1f} "
      f"(Methods: 96 cm from the beginning of the arm to the end of the reward station)")
print(f"fraction of 500 Hz linearized samples that are NaN, per epoch: max {checks.lin_nan_frac.max():.4f}")

In [ ]:
T = pd.concat([r["trials"] for r in R.values()], ignore_index=True)
TP = pd.concat([r["trials_prop"] for r in R.values()], ignore_index=True)
T["dayidx"] = T.groupby("rat").day.transform(lambda s: s.rank(method="dense").astype(int) - 1)
TP["dayidx"] = T["dayidx"].values
T["nses"] = T.groupby(["rat", "day", "epoch"]).trial.transform("size")
TP["nses"] = T["nses"].values
assert (T.start.values == TP.start.values).all()
RATS = ["M", "E", "J", "K", "L"]

# The rewarded cycle per rat, read from the reward log (claims file, opening comment).
seqs = {}
for rat, g in T.groupby("rat"):
    cyc, sup = ct.infer_sequence(g)
    seqs[rat] = cyc
    a = 1
    order = [a]
    for _ in range(4):
        a = cyc[a]
        order.append(a)
    print(f"rat {rat}: rewarded cycle {'-'.join(map(str, order))}; share of rewards followed by the cycle's "
          f"next arm as next reward: {min(sup.values()):.2f} to {max(sup.values()):.2f}")


def add_correct_arm(df):
    """Correct arm of each trial: the cycle's next arm after the last rewarded arm of the session."""
    out = np.full(len(df), np.nan)
    for _, g in df.groupby(["rat", "day", "epoch"], sort=False):
        last = None
        for i, (rat, fin, rew) in zip(g.index, zip(g.rat, g.final, g.rewarded)):
            if last is not None:
                out[i] = seqs[rat][last]
            if rew:
                last = int(fin)
    df["correct_arm"] = out
    return df


T = add_correct_arm(T)
TP["correct_arm"] = T.correct_arm.values
ok = T.correct_arm.notna()
print(f"\nTrials with a correct arm defined (a reward earlier in the session): {ok.sum()} of {len(T)}; "
      f"rewarded exactly when the final arm is the correct arm in {np.mean(T[ok].rewarded == (T[ok].final == T[ok].correct_arm)):.4f}")

The reward log gives 3-4-2-1-3 for rats M, E and J and 1-2-4-3-1 for rats K and L. The Results say
1-2-4-3-1 in two rats and 3-4-2-1-3 in three, which fits; the Figure 1B legend gives 3-4-2-1-3 for
rat E, which fits; Figure S1E labels Seq2 (1-2-4-3-1) for M, E and J, which does not, unless the
arm numbers in the files differ from the figure's. The files label the epochs of K and L "Rev2" and
those of M, E and J "Seq2".

## fig1c-learning

> All rats learned the spatial sequence task over a period of 5 to 19 days, the percentage of
> consecutive trials conforming to the correct sequence transitions rising over the days analysed.

In [ ]:
U = T[T.nses >= ct.MIN_TRIALS].copy()
U["next"] = U.groupby(["rat", "day", "epoch"]).final.shift(-1)
U = U.dropna(subset=["next"])
U["seqtrans"] = np.array([seqs[r][int(a)] == int(b) for r, a, b in zip(U.rat, U.final, U.next)], dtype=float)
lc = U.groupby(["rat", "dayidx"]).seqtrans.mean().unstack(0)[RATS]
print("Fraction of adjacent-trial transitions that are transitions of the rat's cycle, per day (sessions >= 20 trials):")
print(lc.round(3).to_string())
rows = []
for rat in RATS:
    v = lc[rat].dropna()
    rho, p = stats.spearmanr(v.index, v.values)
    rows.append({"rat": rat, "days": len(v), "first": v.iloc[0], "peak_day": int(v.idxmax()), "peak": v.max(),
                 "last_day_is_peak": int(v.idxmax()) == len(v) - 1, "spearman_rho": rho, "spearman_p": p,
                 "holds": (v.max() > v.iloc[0]) and (v.max() > 0.25) and rho > 0})
fig1c = pd.DataFrame(rows)
print(fig1c.round(4).to_string(index=False))
print(f"rats meeting the criterion: {fig1c.holds.sum()} of 5")
fig, ax = plt.subplots(figsize=(5, 3))
for rat in RATS:
    v = lc[rat].dropna()
    ax.plot(v.index - v.idxmax(), 100 * v.values, marker="o", label=rat)
ax.axhline(25, ls="--", c="gray")
ax.set_xlabel("days from peak performance")
ax.set_ylabel("% transitions of the cycle")
ax.legend(fontsize=7)
print(f"plotted range: {100 * lc.min().min():.1f} to {100 * lc.max().max():.1f} %")

**Result: reproduced.** In every rat the value on its peak day is above the first day's and above
25%, and the Spearman correlation with day is positive (table above). The dandiset holds 5, 9, 12,
8 and 6 days for rats M, E, J, K and L, against about 6, 9, 11, 8 and 6 in Figure 1C.

## fig1e-trial-counts

> The dataset analysed holds 2370, 3554, 4628, 3779 and 2087 trials for rats M, E, J, K and L, of
> which 107, 225, 138, 315 and 309 are COM trials, hundreds of COM trials per subject.

In [ ]:
# expected values and tolerances of the claims file
exp_trials = {"M": 2370, "E": 3554, "J": 4628, "K": 3779, "L": 2087}
tol_trials = {"M": 237, "E": 355, "J": 463, "K": 378, "L": 209}
exp_com = {"M": 107, "E": 225, "J": 138, "K": 315, "L": 309}
tol_com = {"M": 11, "E": 23, "J": 14, "K": 32, "L": 31}
readings = []
for theta, df in (("a, 10 cm", T), ("b, 0.1 of arm length", TP)):
    for sel, d in (("i, all trials", df), ("ii, sessions >= 20 trials", df[df.nses >= ct.MIN_TRIALS])):
        n = d.groupby("rat").size()
        c = d.groupby("rat").com.sum()
        within = [abs(n[r] - exp_trials[r]) <= tol_trials[r] for r in RATS] + \
                 [abs(c[r] - exp_com[r]) <= tol_com[r] for r in RATS]
        readings.append({"threshold": theta, "trials": sel,
                         **{f"n_{r}": n[r] for r in RATS}, **{f"com_{r}": int(c[r]) for r in RATS},
                         "within_tol": sum(within), "min_com": int(c.min())})
fig1e = pd.DataFrame(readings)
print("expected trials", exp_trials, "\nexpected COM trials", exp_com)
print(fig1e.to_string(index=False))
print("\nTrials per rat relative to the paper (reading a-i):",
      {r: round(fig1e.iloc[0][f"n_{r}"] / exp_trials[r], 3) for r in RATS})
print("COM trials as a share of trials, dataset (reading a-i) and paper:",
      {r: (round(fig1e.iloc[0][f"com_{r}"] / fig1e.iloc[0][f"n_{r}"], 3), round(exp_com[r] / exp_trials[r], 3)) for r in RATS})

**Result: not reproduced.** The best readings (threshold b) put 4 of the 10 counts within
tolerance, the trial and COM counts of rats E and J; reading a puts 3. The criterion asks for at
least 5 for a partial and 8 for a full reproduction. Every rat has more than 100 COM trials under
every reading (minimum 120). The dandiset holds 0.88, 0.78 and 0.87 of the paper's trials for rats
M, K and L (rat M has 5 days against the paper's 6), and the COM share of trials is close to the
paper's for rats E, J, K and L (0.056 against 0.063, 0.028 against 0.030, 0.076 against 0.083,
0.132 against 0.148), so most of the shortfall is in trials missing from the dandiset; rat M has
more COM trials than the paper (120 and 127 against 107).

## fig1d-initial-switch-bias

> Early in learning every rat was biased to side-switching transitions, with a ratio of switch-side
> to same-side transition probabilities greater than 2 (the ratio for a random choice).

In [ ]:
V = U[U.final != U.next].copy()
V["switch"] = [ct.is_switch(a, b) for a, b in zip(V.final, V.next)]
cnt = V.groupby(["rat", "dayidx"]).switch.agg(["sum", "size"])
cnt["ratio"] = cnt["sum"] / (cnt["size"] - cnt["sum"])
ratio = cnt.ratio.unstack(0)[RATS]
print("P(switch-side)/P(same-side) per day (adjacent trials to different arms, sessions >= 20 trials):")
print(ratio.round(2).to_string())
first = ratio.iloc[0]
first2 = ratio.iloc[:2].mean()
for name, v in (("a, first day", first), ("b, mean of first two days", first2)):
    k = int((v > 2).sum())
    print(f"reading {name}: {v.round(2).to_dict()}; above 2 in {k} of 5; one-sided binomial p = "
          f"{stats.binomtest(k, 5, 0.5, alternative='greater').pvalue:.3f}")
print("Transitions excluded because both trials went to the same arm:", int((U.final == U.next).sum()), "of", len(U))

**Result: reproduced.** On the first analysed day the ratio is 5.53, 4.23, 3.76, 2.31 and 2.25
for rats M, E, J, K and L, above 2 in all five (one-sided binomial p = 0.031); the mean of the
first two days is also above 2 in all five.

## fig1d-switch-bias-declines

> As learning progressed the switch-side to same-side ratio fell to about 1.5, with each day in the
> maze multiplying the bias by an estimated 0.92.

In [ ]:
X = pd.get_dummies(V.rat).astype(float)[RATS]
X["day"] = V.dayidx.astype(float)
m = sm.Logit(V.switch.astype(float), X).fit(disp=0)
ci = np.exp(m.conf_int().loc["day"]).values
print(f"(a/b, i) logistic regression on day index with one indicator per rat, no intercept: n = {len(V)} transitions; "
      f"day effect exp(b) = {np.exp(m.params['day']):.4f} (95% CI {ci[0]:.4f} to {ci[1]:.4f}), p = {m.pvalues['day']:.2e}")
X2 = X.copy()
X2["day"] = (V.dayidx - V.groupby("rat").dayidx.transform("max")).astype(float)
m2 = sm.Logit(V.switch.astype(float), X2).fit(disp=0)
print(f"(a) days from the last day: exp(b) = {np.exp(m2.params['day']):.4f}, p = {m2.pvalues['day']:.2e}")
Xb = sm.add_constant(X.drop(columns="M"))
mb = sm.Logit(V.switch.astype(float), Xb).fit(disp=0)
print(f"intercept and four rat indicators: exp(b) = {np.exp(mb.params['day']):.4f}, p = {mb.pvalues['day']:.2e}")
for rat in RATS:
    v = V[V.rat == rat]
    mr = sm.Logit(v.switch.astype(float), sm.add_constant(v.dayidx.astype(float))).fit(disp=0)
    print(f"  rat {rat} alone: exp(b) = {np.exp(mr.params['dayidx']):.3f}, p = {mr.pvalues['dayidx']:.2e}")
peak = {r: int(lc[r].dropna().idxmax()) for r in RATS}
pr = {r: ratio.loc[peak[r], r] for r in RATS}
pooled_peak = V[[d == peak[r] for r, d in zip(V.rat, V.dayidx)]]
pooled_ratio = pooled_peak.switch.sum() / (~pooled_peak.switch).sum()
print(f"\nratio on each rat's peak day: {dict((r, round(v, 2)) for r, v in pr.items())}")
print(f"final ratio (a) mean over rats = {np.mean(list(pr.values())):.3f}; (b) pooled over rats = {pooled_ratio:.3f}")
last = {r: ratio[r].dropna().iloc[-1] for r in RATS}
print(f"ratio on each rat's last day: {dict((r, round(v, 2)) for r, v in last.items())}, mean {np.mean(list(last.values())):.3f}")

**Result: partially reproduced.** Each day multiplies the odds of a switch-side transition by
0.933 (95% CI 0.920 to 0.946, p = 2.3e-23), within 0.92 plus or minus 0.092, under every coding of
the day index and the rat indicators. The ratio on the peak day is 1.85 as a mean over rats and
1.84 pooled, outside 1.5 plus or minus 0.15 under both readings. The peak day is each rat's last
day in the dandiset. Rat K alone shows no significant decline (0.984, p = 0.38).

## figs1f-multiple-com

> In most COM trials the rats changed their minds once (entering two choice arms), but trials with
> several COM events (more than two arms entered) occurred in every rat.

In [ ]:
for name, df in (("a, 10 cm", T), ("b, 0.1 of arm length", TP)):
    c = df[df.com]
    tab = c.groupby("rat").agg(com_trials=("com", "size"), one_event=("n_com", lambda s: (s == 1).mean()),
                               two_plus_events=("n_com", lambda s: int((s >= 2).sum())),
                               two_plus_arms=("n_com_arm", lambda s: int((s >= 2).sum())),
                               max_events=("n_com", "max")).loc[RATS]
    tab["holds"] = (tab.one_event > 0.5) & (tab.two_plus_events >= 1)
    print(f"reading {name}:")
    print(tab.round(3).to_string())

**Result: reproduced.** In every rat more than half of COM trials have one COM event, and every rat
has trials with two or more, under both thresholds (tables above); trials with more than two arms
entered (two or more COM arms) also occur in every rat.

## fig1g-com-corrective

> COM trials were corrective; the final choice of a COM trial was rewarded more often than the
> initial choice would have been, in each rat.

In [ ]:
def corrective(df, label):
    c = df[df.com & df.correct_arm.notna()].copy()
    c["init_correct"] = (c.initial == c.correct_arm).astype(float)
    c["final_correct"] = c.rewarded.astype(float)
    daily = c.groupby(["rat", "dayidx"])[["init_correct", "final_correct"]].mean()
    rows = []
    for rat in RATS:
        d = daily.loc[rat]
        p = stats.ranksums(d.final_correct, d.init_correct).pvalue
        rows.append({"rat": rat, "days": len(d), "com_trials": int((c.rat == rat).sum()),
                     "initial_median": d.init_correct.median(), "final_median": d.final_correct.median(),
                     "final_higher": d.final_correct.median() > d.init_correct.median(), "ranksum_p": p})
    out = pd.DataFrame(rows)
    out["holds"] = out.final_higher & (out.ranksum_p < 0.05)
    print(f"reading {label} (COM trials with a correct arm defined; dropped as first in session before a reward: "
          f"{int((df.com & df.correct_arm.isna()).sum())}):")
    print(out.round(4).to_string(index=False))
    return out, daily


g1, daily_g = corrective(T, "a, 10 cm")
g1b, _ = corrective(TP, "b, 0.1 of arm length")
print(f"rats meeting the criterion: {g1.holds.sum()} (10 cm), {g1b.holds.sum()} (0.1 of arm)")

**Result: reproduced.** In all five rats the daily probability that the final choice was rewarded
is higher than that of the initial choice (medians 0.47 to 0.72 against 0.00 to 0.18), with
rank-sum p from 0.028 (rat M) down to below 1e-4 (rat J), the same under both thresholds.

## fig1g-com-matches-run-through

> After correction, the reward rate on COM trials did not differ from that of temporally nearby
> run-through trials.

In [ ]:
rng = np.random.default_rng(1836)
sessions = [(k, g.reset_index()) for k, g in T.groupby(["rat", "day", "epoch"], sort=True)]
pvals = {r: [] for r in RATS}
diffs = {r: [] for r in RATS}
no_ctrl = 0
for draw in range(100):
    rows = []
    for (rat, day, ep), g in sessions:
        if not g.com.any():
            continue
        ctrl = ct.nearby_control(g, rng)
        comrows = g[g.com]
        for (_, cr), j in zip(comrows.iterrows(), ctrl):
            rows.append({"rat": rat, "dayidx": cr.dayidx, "com_rew": float(cr.rewarded),
                         "ctrl_rew": float(g.rewarded.iloc[j]) if j >= 0 else np.nan})
    d = pd.DataFrame(rows)
    if draw == 0:
        no_ctrl = int(d.ctrl_rew.isna().sum())
    com_daily = d.groupby(["rat", "dayidx"]).com_rew.mean()
    ctrl_daily = d.dropna().groupby(["rat", "dayidx"]).ctrl_rew.mean()
    for rat in RATS:
        a, b = com_daily.loc[rat], ctrl_daily.loc[rat]
        pvals[rat].append(stats.ranksums(a, b).pvalue)
        diffs[rat].append(a.median() - b.median())
res = pd.DataFrame({"rat": RATS, "median_p": [np.median(pvals[r]) for r in RATS],
                    "share_p_below_0.05": [np.mean(np.array(pvals[r]) < 0.05) for r in RATS],
                    "median_diff_COM_minus_nearby": [np.median(diffs[r]) for r in RATS]})
print(f"COM trials without an eligible nearby trial (draw 0): {no_ctrl}")
print(res.round(4).to_string(index=False))
print("rats with a significant difference (median p < 0.05):", int((res.median_p < 0.05).sum()))

**Result: reproduced.** No rat shows a significant difference between the daily reward rate of
COM trials and that of nearby non-COM trials: the median p over 100 draws is 0.19 to 0.75, and at
most 16% of draws (rat K) give p < 0.05. The median differences are -0.06 to 0.12.

## fig1h-final-choice-less-switching

> The initial choice on COM trials was heavily biased to side-switching, while the final choice
> was much less so, with a switch-side to same-side ratio close to 1.

In [ ]:
def sign_p(x, ref):
    x = np.asarray(x)
    above, below = int((x > ref).sum()), int((x < ref).sum())
    return stats.binomtest(above, above + below, 0.5).pvalue if above + below else np.nan, above, below


def fig1h_table(df, label):
    d = df.copy()
    d["prev"] = d.groupby(["rat", "day", "epoch"]).final.shift(1)
    c = d[d.com & d.prev.notna()].copy()
    rows = []
    for kind in ("initial", "final"):
        x = c[c[kind] != c.prev]
        x = x.assign(sw=[ct.is_switch(a, b) for a, b in zip(x.prev, x[kind])])
        cnt = x.groupby(["rat", "dayidx"]).sw.agg(["sum", "size"])
        cnt["ratio"] = cnt["sum"] / (cnt["size"] - cnt["sum"])
        rows.append(cnt.ratio.rename(kind))
    daily = pd.concat(rows, axis=1).replace(np.inf, np.nan)
    out = []
    for rat in RATS:
        dd = daily.loc[rat]
        both = dd.dropna()
        t = stats.ttest_rel(both.initial, both.final) if len(both) > 1 else None
        pi2, ia, ib = sign_p(dd.initial.dropna(), 2)
        pf2, fa, fb = sign_p(dd.final.dropna(), 2)
        pf1, f1a, f1b = sign_p(dd.final.dropna(), 1)
        out.append({"rat": rat, "days_both": len(both), "dropped_days_init": int(dd.initial.isna().sum()),
                    "dropped_days_final": int(dd.final.isna().sum()),
                    "init_median": dd.initial.median(), "final_median": dd.final.median(),
                    "paired_t_p": t.pvalue if t else np.nan, "init_gt_final": both.initial.mean() > both.final.mean(),
                    "sign_init_vs2_p": pi2, "init_above2": f"{ia}/{ia + ib}",
                    "sign_final_vs2_p": pf2, "final_below2": f"{fb}/{fa + fb}",
                    "sign_final_vs1_p": pf1, "final_above1": f"{f1a}/{f1a + f1b}"})
    out = pd.DataFrame(out)
    print(f"reading {label}:")
    print(out.round(4).to_string(index=False))
    t_ok = int(((out.paired_t_p < 0.01) & out.init_gt_final).sum())
    i_ok = int(((out.sign_init_vs2_p < 0.05) & (out.init_median > 2)).sum())
    f_ok = int(((out.sign_final_vs2_p < 0.05) & (out.final_median < 2)).sum())
    print(f"paired t p < 0.01 with initial > final: {t_ok} rats; initial above 2 by sign test: {i_ok}; "
          f"final below 2 by sign test: {f_ok}")
    return out, daily


h1, daily_h = fig1h_table(T, "a, 10 cm")
h1b, _ = fig1h_table(TP, "b, 0.1 of arm length")

**Result: not reproduced.** The initial ratio exceeds the final in all five rats, but the paired
t-test reaches p < 0.01 only in rats J and K (p = 0.0036 and 0.0040 at 10 cm); rats E and L have p
= 0.013 and the criterion asks for four rats at p < 0.01, or three for a partial reproduction. The
sign tests hold as the legend reports: the initial ratio is above 2 and the final below 2 with p <
0.05 in rats E, J, K and L, not in rat M. The 0.1-of-arm threshold gives the same counts.

## fig1h-final-ratio-near-one

> The final choice of COM trials did not differ from a switch-side to same-side ratio of 1 in any
> rat.

In [ ]:
print(h1[["rat", "final_median", "final_above1", "sign_final_vs1_p"]].round(4).to_string(index=False))
print("rats differing from 1 (sign test p < 0.05), 10 cm:", int((h1.sign_final_vs1_p < 0.05).sum()),
      "; 0.1 of arm:", int((h1b.sign_final_vs1_p < 0.05).sum()))

**Result: reproduced.** No rat's daily final-choice ratio differs from 1 by the sign test (p =
0.18 to 1.0); the medians are 0.97 to 1.38.

## Blocked claims

The claims on hippocampal content (Figures 3 to 5, S3, S6 to S8, S10, S11) need the clusterless
state-space decoder's posterior per 2 ms bin, its continuous-state probability, and events built on
it. None of these is in the dandiset; the decoder would have to be refit from the 30 kHz voltage
(12.5 TB of raw files, 313 GB per day on average) after spike detection, which does not fit an entry's time
limit. The analysis files with `scratch/triggered` tables hold decoded and head linear positions
around unnamed trigger times, with an undocumented `triggered_trial_info` column; nothing says
which events, trials or windows they hold. The cell below lists them.

In [ ]:
print("triggered tables per rat:", kinds.groupby("rat").triggered.sum().to_dict())
lfp_assets = assets[assets.kind == "lfp"]
print(f"LFP files: {len(lfp_assets)}, {lfp_assets['size'].sum() / 1e9:.0f} GB; per rat {lfp_assets.groupby('rat').size().to_dict()}")
f = ct.open_nwb(lfp_assets.iloc[0].url)
es = f["scratch/filtered data"]
t0 = time.time()
block = es["data"][:600000, :]
dt = time.time() - t0
print(f"{lfp_assets.iloc[0].path}: {es['data'].shape} samples x channels, described as '{es.attrs['description']}'; "
      f"reading 600 s took {dt:.1f} s ({block.nbytes / 1e6 / dt:.0f} MB/s)")
raw_assets = assets[assets.kind == "raw"]
print(f"raw files: {raw_assets['size'].sum() / 1e12:.1f} TB, {raw_assets['size'].mean() / 1e9:.0f} GB per day on average")
pos = assets[assets.kind == "obj_behavior"].iloc[0]
ho = ct.open_nwb(pos.url)["scratch/CompassDirection/head_orientation/data"]
print(f"head orientation in {pos.path}: unit '{ho.attrs['unit']}', range {np.nanmin(ho[:]):.2f} to {np.nanmax(ho[:]):.2f}")
print(f"elapsed {time.time() - started:.0f} s")

## Summary

| Claim | Outcome |
|---|---|
| fig1c-learning | reproduced |
| fig1e-trial-counts | not reproduced |
| fig1d-initial-switch-bias | reproduced |
| fig1d-switch-bias-declines | partially reproduced |
| figs1f-multiple-com | reproduced |
| fig1g-com-corrective | reproduced |
| fig1g-com-matches-run-through | reproduced |
| fig1h-final-choice-less-switching | not reproduced |
| fig1h-final-ratio-near-one | reproduced |
| fig3b-extended-before-stopping, fig3b-distance-predicts-com, fig3c-distant-after-stopping, fig3d-distant-vs-inbound | blocked, metadata insufficient |
| the other 16 claims on hippocampal content | blocked, too large |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

How the analysis differs from the paper's. Trials are built from the digital poke and pump
events, and agree with the Statescript logs to within one trial per session. COM trials are
detected on the deposited 500 Hz linearization with the Methods' rule; both thresholds of the
claims file are run and give the same outcomes for every claim but the counts. The closing
structure of the reversal detection is a standard choice. The per-rat sequence is read from the
reward log. Each rat's peak day is its last day in the dandiset.

What was found about the dataset. The dandiset holds fewer trials than the paper reports for
rats M (5 days against 6 in the paper), K and L. Rat M's 500 Hz linearizations are each deposited
twice, as identical copies, and its Statescript logs are named "syayescript". None of the Spyglass
analysis files (2415 of 2494 assets) says what it holds; the triggered decode tables cannot be
interpreted without a description. The decoder output that the neural claims rest on is not
deposited. The reward log gives the sequence 3-4-2-1-3 for M, E and J and 1-2-4-3-1 for K and L,
against Figure S1E's labels.